# M5 FOODS Demand Forecasting

M5 Forecasting Accuracy の FOODS カテゴリを対象に、販売規模別の需要予測を行います。

- 商品選定は最初の評価日 `2015-10-01` より前の販売履歴だけを使用
- `top` / `middle` / `bottom` から各30商品を固定抽出
- lag / rolling には予測対象日の販売数を含めない
- 最終ホールドアウトと walk-forward の両方で評価
- 特徴量セットと目的関数・目的変数変換を比較


In [1]:
from __future__ import annotations

from pathlib import Path
import sys
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.item_grouping import apply_item_groups, select_item_groups

DATA_DIRS = [
    PROJECT_ROOT / "data" / "raw",
    PROJECT_ROOT / "data",
    PROJECT_ROOT / "input",
    PROJECT_ROOT / "m5-forecasting-accuracy",
]
RESULTS_DIR = PROJECT_ROOT / "results"
IMAGES_DIR = PROJECT_ROOT / "images"
MODELS_DIR = PROJECT_ROOT / "models"
for directory in (RESULTS_DIR, IMAGES_DIR, MODELS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

GROUPS = ["top", "middle", "bottom"]
GROUPING_CUTOFF_DATE = pd.Timestamp("2015-10-01")
FINAL_SPLIT_DATE = pd.Timestamp("2016-03-01")
ITEMS_PER_GROUP = 30
RANDOM_STATE = 42


## 1. データ読み込み

In [2]:
def find_data_file(filename: str) -> Path:
    for base in DATA_DIRS:
        direct = base / filename
        if direct.is_file():
            return direct
        if base.is_dir():
            matches = list(base.rglob(filename))
            if matches:
                return matches[0]
    searched = ", ".join(str(path) for path in DATA_DIRS)
    raise FileNotFoundError(
        f"{filename} was not found. Place M5 files under one of: {searched}"
    )

sales_path = find_data_file("sales_train_validation.csv")
calendar_path = find_data_file("calendar.csv")
prices_path = find_data_file("sell_prices.csv")

sales = pd.read_csv(sales_path)
calendar = pd.read_csv(calendar_path)
prices = pd.read_csv(prices_path)
calendar["date"] = pd.to_datetime(calendar["date"], errors="raise")

print("sales:", sales.shape)
print("calendar:", calendar.shape)
print("prices:", prices.shape)


sales: (30490, 1919)
calendar: (1969, 14)
prices: (6841121, 4)


## 2. 評価開始前の履歴だけで商品グループを固定

In [3]:
foods = sales.loc[sales["cat_id"].eq("FOODS")].copy()
selected_items = select_item_groups(
    foods,
    calendar,
    cutoff_date=GROUPING_CUTOFF_DATE,
    items_per_group=ITEMS_PER_GROUP,
    output_path=RESULTS_DIR / "selected_item_groups.csv",
)
grouped_sales = apply_item_groups(foods, selected_items)

display(
    selected_items.groupby("sales_group", observed=True)
    .agg(
        item_count=("item_id", "nunique"),
        min_total=("total_sales_before_cutoff", "min"),
        median_total=("total_sales_before_cutoff", "median"),
        max_total=("total_sales_before_cutoff", "max"),
    )
    .reindex(GROUPS)
)


,item_count,min_total,median_total,max_total
sales_group,,,,
top,30,177281,243989.0,907103
middle,30,12535,12764.0,13152
bottom,30,346,944.0,1222


## 3. 縦持ち変換と外部テーブル結合

In [4]:
day_columns = [column for column in grouped_sales.columns if column.startswith("d_")]
id_columns = [
    column for column in [
        "id", "item_id", "dept_id", "cat_id", "store_id", "state_id", "sales_group"
    ] if column in grouped_sales.columns
]

long_df = grouped_sales.melt(
    id_vars=id_columns,
    value_vars=day_columns,
    var_name="d",
    value_name="sales",
)
calendar_columns = [
    column for column in [
        "d", "date", "wm_yr_wk", "weekday", "wday", "month", "year",
        "event_name_1", "event_type_1", "event_name_2", "event_type_2",
        "snap_CA", "snap_TX", "snap_WI",
    ] if column in calendar.columns
]
long_df = long_df.merge(
    calendar[calendar_columns], on="d", how="left", validate="many_to_one"
)
long_df = long_df.merge(
    prices[["store_id", "item_id", "wm_yr_wk", "sell_price"]],
    on=["store_id", "item_id", "wm_yr_wk"],
    how="left",
    validate="many_to_one",
)
long_df["date"] = pd.to_datetime(long_df["date"], errors="raise")
long_df["sales"] = pd.to_numeric(long_df["sales"], errors="coerce").fillna(0.0)
long_df = long_df.sort_values(["id", "date"]).reset_index(drop=True)
print(long_df.shape)


(1721700, 23)


## 4. リークを避けた特徴量生成

In [5]:
def add_features(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.sort_values(["id", "date"]).copy()
    by_series = result.groupby("id", sort=False, observed=True)

    for lag in (7, 14, 21, 28):
        result[f"lag_{lag}"] = by_series["sales"].shift(lag)

    shifted = by_series["sales"].shift(1)
    for window in (7, 14, 28):
        result[f"rolling_mean_{window}"] = (
            shifted.groupby(result["id"], sort=False).rolling(window).mean()
            .reset_index(level=0, drop=True)
        )
        result[f"rolling_std_{window}"] = (
            shifted.groupby(result["id"], sort=False).rolling(window).std()
            .reset_index(level=0, drop=True)
        )
        result[f"zero_rate_{window}"] = (
            shifted.eq(0).groupby(result["id"], sort=False).rolling(window).mean()
            .reset_index(level=0, drop=True)
        )

    result["trend_7_28"] = result["rolling_mean_7"] - result["rolling_mean_28"]
    result["lag_diff_7_28"] = result["lag_7"] - result["lag_28"]

    result["sell_price"] = result.groupby("id", observed=True)["sell_price"].ffill()
    result["price_lag_1"] = result.groupby("id", observed=True)["sell_price"].shift(1)
    result["price_change_1"] = result["sell_price"] - result["price_lag_1"]
    result["price_pct_change_1"] = np.where(
        result["price_lag_1"].ne(0),
        result["price_change_1"] / result["price_lag_1"],
        0.0,
    )
    result["price_up"] = result["price_change_1"].gt(0).astype("int8")
    result["price_down"] = result["price_change_1"].lt(0).astype("int8")
    result["has_event"] = result[["event_name_1", "event_name_2"]].notna().any(axis=1).astype("int8")

    result["is_snap_day"] = 0
    for state, column in {"CA": "snap_CA", "TX": "snap_TX", "WI": "snap_WI"}.items():
        if column in result.columns:
            mask = result["state_id"].eq(state)
            result.loc[mask, "is_snap_day"] = (
                pd.to_numeric(result.loc[mask, column], errors="coerce").fillna(0).astype("int8")
            )

    result["day"] = result["date"].dt.day.astype("int16")
    result["week_of_month"] = ((result["day"] - 1) // 7 + 1).astype("int8")
    result["is_month_start"] = result["date"].dt.is_month_start.astype("int8")
    result["is_month_end"] = result["date"].dt.is_month_end.astype("int8")

    result = pd.get_dummies(
        result,
        columns=["dept_id", "store_id", "state_id"],
        prefix=["dept", "store", "state"],
        dtype="int8",
    )
    return result

feature_df = add_features(long_df)
print(feature_df.shape)


(1721700, 62)


## 5. 評価関数と特徴量セット

In [6]:
BASE_FEATURES = [
    "wday", "month", "year", "day", "week_of_month", "is_month_start", "is_month_end",
    "lag_7", "lag_14", "lag_21", "lag_28",
    "rolling_mean_7", "rolling_mean_14", "rolling_mean_28",
    "rolling_std_7", "rolling_std_14", "rolling_std_28",
    "zero_rate_7", "zero_rate_14", "zero_rate_28",
]
TREND_FEATURES = ["trend_7_28", "lag_diff_7_28"]
PRICE_EVENT_FEATURES = [
    "sell_price", "price_lag_1", "price_change_1", "price_pct_change_1",
    "price_up", "price_down", "has_event", "is_snap_day",
]
DUMMY_FEATURES = [
    column for column in feature_df.columns
    if column.startswith(("dept_", "store_", "state_"))
]

FEATURE_SETS = {
    "base_lag_rolling": BASE_FEATURES,
    "v2_trend": BASE_FEATURES + TREND_FEATURES,
    "v3_dow_effect": BASE_FEATURES + TREND_FEATURES + ["dow_effect", "dow_lag7_interaction"],
    "v4_price_event_dept": BASE_FEATURES + TREND_FEATURES + [
        "dow_effect", "dow_lag7_interaction"
    ] + PRICE_EVENT_FEATURES + DUMMY_FEATURES,
}

REQUIRED_HISTORY_FEATURES = ["lag_28", "rolling_mean_28", "rolling_std_28", "zero_rate_28"]
model_df = feature_df.dropna(subset=REQUIRED_HISTORY_FEATURES).copy()


def add_train_only_dow_effect(
    train: pd.DataFrame, test: pd.DataFrame
) -> tuple[pd.DataFrame, pd.DataFrame]:
    train_result = train.copy()
    test_result = test.copy()
    keys = ["item_id", "wday"]
    lookup = train_result.groupby(keys, observed=True)["sales"].mean().rename("dow_effect")
    fallback = train_result.groupby("wday", observed=True)["sales"].mean()
    global_mean = float(train_result["sales"].mean())

    for target in (train_result, test_result):
        target["dow_effect"] = target.set_index(keys).index.map(lookup)
        target["dow_effect"] = target["dow_effect"].fillna(target["wday"].map(fallback)).fillna(global_mean)
        target["dow_lag7_interaction"] = target["dow_effect"] * target["lag_7"]
    return train_result, test_result


def metrics(y_true: pd.Series, predictions: np.ndarray) -> dict[str, float]:
    predictions = np.clip(np.asarray(predictions, dtype=float), 0.0, None)
    mae = mean_absolute_error(y_true, predictions)
    rmse = np.sqrt(mean_squared_error(y_true, predictions))
    mean_sales = float(np.mean(y_true))
    return {
        "mae": float(mae),
        "rmse": float(rmse),
        "mean_sales": mean_sales,
        "mae_ratio": float(mae / mean_sales) if mean_sales > 0 else np.nan,
    }


def build_model(objective: str = "reg:squarederror") -> XGBRegressor:
    return XGBRegressor(
        objective=objective,
        n_estimators=300,
        max_depth=8,
        learning_rate=0.05,
        subsample=0.85,
        colsample_bytree=0.85,
        min_child_weight=5,
        reg_lambda=1.0,
        tree_method="hist",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )


def prepare_xy(
    frame: pd.DataFrame, feature_names: list[str]
) -> tuple[pd.DataFrame, pd.Series]:
    existing = [feature for feature in feature_names if feature in frame.columns]
    X = frame[existing].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    y = frame["sales"].astype(float)
    return X, y


## 6. 最終時系列分割とベースライン

In [7]:
train_all = model_df.loc[model_df["date"] < FINAL_SPLIT_DATE].copy()
test_all = model_df.loc[model_df["date"] >= FINAL_SPLIT_DATE].copy()

baseline_rows = []
for group in GROUPS:
    test_group = test_all.loc[test_all["sales_group"].eq(group)].copy()
    for baseline_name, column in {
        "seasonal_naive_lag_7": "lag_7",
        "rolling_mean_7": "rolling_mean_7",
        "rolling_mean_28": "rolling_mean_28",
    }.items():
        row = {"sales_group": group, "model": baseline_name}
        row.update(metrics(test_group["sales"], test_group[column].fillna(0.0).to_numpy()))
        baseline_rows.append(row)

baseline_results = pd.DataFrame(baseline_rows)
baseline_results.to_csv(RESULTS_DIR / "m5_baseline_results.csv", index=False)
display(baseline_results)


,sales_group,model,mae,rmse,mean_sales,mae_ratio
0,top,seasonal_naive_lag_7,7.007818,11.837979,15.934182,0.439798
1,top,rolling_mean_7,5.972623,9.998217,15.934182,0.374831
2,top,rolling_mean_28,6.420537,10.380675,15.934182,0.402941
3,middle,seasonal_naive_lag_7,1.036303,1.714342,0.922909,1.122866
4,middle,rolling_mean_7,0.872320,1.314975,0.922909,0.945186
5,middle,rolling_mean_28,0.876513,1.288903,0.922909,0.949728
6,bottom,seasonal_naive_lag_7,0.556303,1.106099,0.415212,1.339804
7,bottom,rolling_mean_7,0.488398,0.832750,0.415212,1.176262
8,bottom,rolling_mean_28,0.491803,0.816249,0.415212,1.184462


## 7. Ablation Study

In [8]:
feature_compare_rows = []
feature_models = {}

for group in GROUPS:
    train_group = train_all.loc[train_all["sales_group"].eq(group)].copy()
    test_group = test_all.loc[test_all["sales_group"].eq(group)].copy()
    train_group, test_group = add_train_only_dow_effect(train_group, test_group)

    for feature_version, feature_names in FEATURE_SETS.items():
        X_train, y_train = prepare_xy(train_group, feature_names)
        X_test, y_test = prepare_xy(test_group, list(X_train.columns))
        model = build_model()
        model.fit(X_train, y_train)
        prediction = model.predict(X_test)
        row = {
            "sales_group": group,
            "feature_version": feature_version,
            "feature_count": X_train.shape[1],
        }
        row.update(metrics(y_test, prediction))
        feature_compare_rows.append(row)
        feature_models[(group, feature_version)] = (model, list(X_train.columns))

feature_compare = pd.DataFrame(feature_compare_rows)
feature_compare.to_csv(RESULTS_DIR / "m5_feature_compare_summary.csv", index=False)
display(feature_compare.sort_values(["sales_group", "mae"]))


,sales_group,feature_version,feature_count,mae,rmse,mean_sales,mae_ratio
11,bottom,v4_price_event_dept,48,0.489756,0.796706,0.415212,1.179531
10,bottom,v3_dow_effect,24,0.492804,0.803741,0.415212,1.186873
9,bottom,v2_trend,22,0.494253,0.804432,0.415212,1.190363
8,bottom,base_lag_rolling,20,0.495413,0.804456,0.415212,1.193156
7,middle,v4_price_event_dept,48,0.852304,1.253918,0.922909,0.923497
6,middle,v3_dow_effect,24,0.853408,1.258189,0.922909,0.924694
4,middle,base_lag_rolling,20,0.854569,1.258253,0.922909,0.925952
5,middle,v2_trend,22,0.854937,1.259258,0.922909,0.926351
3,top,v4_price_event_dept,48,5.256141,8.472408,15.934182,0.329866
2,top,v3_dow_effect,24,5.330262,8.646747,15.934182,0.334517


## 8. squarederror / log1p / Poisson 比較

In [9]:
FINAL_FEATURE_VERSION = "v4_price_event_dept"
final_compare_rows = []
log_models = {}
importance_rows = []

for group in GROUPS:
    train_group = train_all.loc[train_all["sales_group"].eq(group)].copy()
    test_group = test_all.loc[test_all["sales_group"].eq(group)].copy()
    train_group, test_group = add_train_only_dow_effect(train_group, test_group)
    X_train, y_train = prepare_xy(train_group, FEATURE_SETS[FINAL_FEATURE_VERSION])
    X_test, y_test = prepare_xy(test_group, list(X_train.columns))

    configurations = [
        ("XGBoost_squarederror_v4", "reg:squarederror", False),
        ("XGBoost_log1p", "reg:squarederror", True),
        ("XGBoost_poisson", "count:poisson", False),
    ]
    for model_name, objective, use_log_target in configurations:
        model = build_model(objective)
        target = np.log1p(y_train) if use_log_target else y_train
        model.fit(X_train, target)
        raw_prediction = model.predict(X_test)
        prediction = np.expm1(raw_prediction) if use_log_target else raw_prediction
        row = {"sales_group": group, "model": model_name, "feature_version": FINAL_FEATURE_VERSION}
        row.update(metrics(y_test, prediction))
        final_compare_rows.append(row)

        if model_name == "XGBoost_log1p":
            log_models[group] = model
            joblib.dump(model, MODELS_DIR / f"xgboost_log1p_{group}.joblib")
            for feature, importance in zip(X_train.columns, model.feature_importances_):
                importance_rows.append(
                    {"sales_group": group, "feature": feature, "importance": float(importance)}
                )

final_compare = pd.DataFrame(final_compare_rows)
final_compare.to_csv(RESULTS_DIR / "m5_final_model_compare.csv", index=False)
final_best = (
    final_compare.sort_values("mae")
    .groupby("sales_group", observed=True, as_index=False)
    .first()
)
final_best.to_csv(RESULTS_DIR / "m5_final_best_by_group.csv", index=False)

importance_df = pd.DataFrame(importance_rows).sort_values(
    ["sales_group", "importance"], ascending=[True, False]
)
importance_df.to_csv(RESULTS_DIR / "m5_best_feature_importance.csv", index=False)

# 互換用サマリー
final_compare.loc[final_compare["model"].eq("XGBoost_log1p")].to_csv(
    RESULTS_DIR / "m5_log1p_summary.csv", index=False
)
final_compare.loc[final_compare["model"].eq("XGBoost_poisson")].to_csv(
    RESULTS_DIR / "m5_poisson_summary.csv", index=False
)
final_best.to_csv(RESULTS_DIR / "m5_best_summary.csv", index=False)

display(final_compare.sort_values(["sales_group", "mae"]))


,sales_group,model,feature_version,mae,rmse,mean_sales,mae_ratio
7,bottom,XGBoost_log1p,v4_price_event_dept,0.458287,0.821711,0.415212,1.103742
8,bottom,XGBoost_poisson,v4_price_event_dept,0.484593,0.799835,0.415212,1.167098
6,bottom,XGBoost_squarederror_v4,v4_price_event_dept,0.489756,0.796706,0.415212,1.179531
4,middle,XGBoost_log1p,v4_price_event_dept,0.808740,1.275600,0.922909,0.876294
5,middle,XGBoost_poisson,v4_price_event_dept,0.848174,1.249084,0.922909,0.919023
3,middle,XGBoost_squarederror_v4,v4_price_event_dept,0.852304,1.253918,0.922909,0.923497
1,top,XGBoost_log1p,v4_price_event_dept,5.161418,8.926821,15.934182,0.323921
2,top,XGBoost_poisson,v4_price_event_dept,5.240464,8.498971,15.934182,0.328882
0,top,XGBoost_squarederror_v4,v4_price_event_dept,5.256141,8.472408,15.934182,0.329866


## 9. Walk-forward評価

In [10]:
WALK_FORWARD_DATES = [
    pd.Timestamp("2015-10-01"),
    pd.Timestamp("2015-12-01"),
    pd.Timestamp("2016-02-01"),
    pd.Timestamp("2016-03-01"),
]
HORIZON_DAYS = 28
walk_rows = []

for split_date in WALK_FORWARD_DATES:
    split_end = split_date + pd.Timedelta(days=HORIZON_DAYS)
    wf_train_all = model_df.loc[model_df["date"] < split_date].copy()
    wf_test_all = model_df.loc[
        (model_df["date"] >= split_date) & (model_df["date"] < split_end)
    ].copy()

    for group in GROUPS:
        wf_train = wf_train_all.loc[wf_train_all["sales_group"].eq(group)].copy()
        wf_test = wf_test_all.loc[wf_test_all["sales_group"].eq(group)].copy()
        if wf_train.empty or wf_test.empty:
            continue
        wf_train, wf_test = add_train_only_dow_effect(wf_train, wf_test)
        X_train, y_train = prepare_xy(wf_train, FEATURE_SETS[FINAL_FEATURE_VERSION])
        X_test, y_test = prepare_xy(wf_test, list(X_train.columns))
        model = build_model()
        model.fit(X_train, np.log1p(y_train))
        prediction = np.expm1(model.predict(X_test))
        row = {
            "split_date": split_date.date().isoformat(),
            "test_end_date": (split_end - pd.Timedelta(days=1)).date().isoformat(),
            "sales_group": group,
            "model": "XGBoost_log1p",
        }
        row.update(metrics(y_test, prediction))
        walk_rows.append(row)

walk_results = pd.DataFrame(walk_rows)
walk_results.to_csv(RESULTS_DIR / "m5_walk_forward_results.csv", index=False)
walk_summary = (
    walk_results.groupby("sales_group", observed=True)
    .agg(
        evaluation_count=("mae", "size"),
        mean_mae=("mae", "mean"),
        std_mae=("mae", "std"),
        mean_rmse=("rmse", "mean"),
        mean_mae_ratio=("mae_ratio", "mean"),
    )
    .reset_index()
)
walk_summary.to_csv(RESULTS_DIR / "m5_walk_forward_summary.csv", index=False)

for group in GROUPS:
    plot_data = walk_results.loc[walk_results["sales_group"].eq(group)].copy()
    if plot_data.empty:
        continue
    plot_data["split_date"] = pd.to_datetime(plot_data["split_date"])
    ax = plot_data.plot(x="split_date", y="mae_ratio", marker="o", figsize=(9, 5), legend=False)
    ax.set_title(f"Walk-forward MAE ratio: {group}")
    ax.set_xlabel("Evaluation start date")
    ax.set_ylabel("MAE ratio")
    ax.figure.tight_layout()
    ax.figure.savefig(IMAGES_DIR / f"walk_forward_mae_ratio_{group}.png", dpi=150, bbox_inches="tight")
    plt.close(ax.figure)

display(walk_results)
display(walk_summary)


,split_date,test_end_date,sales_group,model,mae,rmse,mean_sales,mae_ratio
0,2015-10-01,2015-10-28,top,XGBoost_log1p,5.389368,9.525147,16.526905,0.326097
1,2015-10-01,2015-10-28,middle,XGBoost_log1p,0.806783,1.353725,0.986667,0.817685
2,2015-10-01,2015-10-28,bottom,XGBoost_log1p,0.393146,0.701097,0.335238,1.172737
3,2015-12-01,2015-12-28,top,XGBoost_log1p,4.286784,9.518303,12.178452,0.351997
4,2015-12-01,2015-12-28,middle,XGBoost_log1p,0.853359,1.402358,0.983095,0.868033
5,2015-12-01,2015-12-28,bottom,XGBoost_log1p,0.409598,0.734901,0.350357,1.169086
6,2016-02-01,2016-02-28,top,XGBoost_log1p,5.537430,10.022624,16.479881,0.336012
7,2016-02-01,2016-02-28,middle,XGBoost_log1p,0.892223,1.422222,1.090238,0.818374
8,2016-02-01,2016-02-28,bottom,XGBoost_log1p,0.445423,0.785492,0.398690,1.117216
9,2016-03-01,2016-03-28,top,XGBoost_log1p,5.399091,9.378539,16.128095,0.334763


,sales_group,evaluation_count,mean_mae,std_mae,mean_rmse,mean_mae_ratio
0,bottom,4,0.423888,0.026865,0.764444,1.141984
1,middle,4,0.840359,0.040682,1.365502,0.847948
2,top,4,5.153168,0.581534,9.611153,0.337217


## 10. 補助集計

In [11]:
eda = long_df.copy()
eda["week_of_month"] = ((eda["date"].dt.day - 1) // 7 + 1).astype("int8")
eda["has_event"] = eda[["event_name_1", "event_name_2"]].notna().any(axis=1)
eda["price_change"] = eda.groupby("id", observed=True)["sell_price"].diff()
eda["price_down"] = eda["price_change"].lt(0)
eda["is_snap_day"] = 0
for state, column in {"CA": "snap_CA", "TX": "snap_TX", "WI": "snap_WI"}.items():
    if column in eda.columns:
        mask = eda["state_id"].eq(state)
        eda.loc[mask, "is_snap_day"] = pd.to_numeric(eda.loc[mask, column], errors="coerce").fillna(0)

eda.groupby(["sales_group", "dept_id"], observed=True)["sales"].agg(["mean", "sum"]).reset_index().to_csv(
    RESULTS_DIR / "m5_dept_sales.csv", index=False
)
eda.groupby(["sales_group", "has_event"], observed=True)["sales"].agg(["mean", "sum"]).reset_index().to_csv(
    RESULTS_DIR / "m5_event_sales.csv", index=False
)
eda.groupby(["sales_group", "is_snap_day"], observed=True)["sales"].agg(["mean", "sum"]).reset_index().to_csv(
    RESULTS_DIR / "m5_snap_sales.csv", index=False
)
eda.groupby(["sales_group", "price_down"], observed=True)["sales"].agg(["mean", "sum"]).reset_index().to_csv(
    RESULTS_DIR / "m5_price_down_sales.csv", index=False
)
eda.groupby(["sales_group", "week_of_month"], observed=True)["sales"].agg(["mean", "sum"]).reset_index().to_csv(
    RESULTS_DIR / "m5_week_of_month_sales.csv", index=False
)

print("Completed. Results:", RESULTS_DIR)
print("Models:", MODELS_DIR)
print("Images:", IMAGES_DIR)


Completed. Results: C:\Users\master\Documents\m5-foods-demand-forecasting-new\results
Models: C:\Users\master\Documents\m5-foods-demand-forecasting-new\models
Images: C:\Users\master\Documents\m5-foods-demand-forecasting-new\images
